# 22 — Δείγμα και ανθρώπινες ετικέτες για decision models

Ετοιμάζει **500 πραγματικές ολόκληρες κριτικές** από το corpus που υπάρχει ήδη στο repo,
χωρισμένες πριν τις ετικέτες σε **300 train / 100 dev / 100 test**.
Κατεβάζει κενά CSV: Α (500), Β (100 ανεξάρτητες κρίσεις), τελικό (500).

**Colab CPU. Δεν χρειάζεται GPU, API key, Drive ή άλλο αρχείο.**
Αυτό το notebook δεν εκπαιδεύει ούτε καλεί μοντέλα και δεν παράγει macro-F1.
Η σύγκριση logistic regression / Qwen / Microsoft-Decision-1 ακολουθεί αφού
κλείσουν οι ανθρώπινες ετικέτες. Το Space και το notebook 21 δεν ενημερώνονται.

Πληθυσμός: δύο ξενοδοχεία NYC, 2008–2009. Το αποτέλεσμα δεν θα είναι εκτίμηση
αξιοπιστίας για όλα τα ξενοδοχεία ή για τις προτάσεις βελτίωσης.
[Πρωτόκολλο](https://github.com/krimits/hotel-review-nlp/blob/claude/funny-rubin-31xgtb/docs/experiments/decision_topic_benchmark/PROTOCOL.md).

## 1. Καρφιτσωμένος κώδικας
Ξεκίνα με νέα συνεδρία CPU και τρέξε τα κελιά με τη σειρά.

In [ ]:
from pathlib import Path
SOURCE_COMMIT = '4d9a1691836354fe4ab8ec0e13b78947fd28f6a0'
BASE = Path("/content/hotel-decision-topics-v1")
REPO = BASE / "repo"
ENV = BASE / "env"
OUT = REPO / "runs/decision_topics/sample"
PYTHON = ENV / "bin/python"
REPO_URL = "https://github.com/krimits/hotel-review-nlp.git"


## 2. Απομονωμένο CPU περιβάλλον
Η εγκατάσταση γίνεται σε νέο venv, χωρίς τα global πακέτα του Colab.
Ο έλεγχος διαβάζει τις πραγματικές εκδόσεις και τη θέση των imports.

In [ ]:
import json
import os
import subprocess
import venv

BASE.mkdir(parents=True, exist_ok=True)
clean_env = {k: v for k, v in os.environ.items()
             if not k.startswith(("PYTHON", "PIP"))
             and k not in {"OPENROUTER_API_KEY", "TYPESAFE_API_KEY", "HF_TOKEN"}}
logs = []
def run(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd, env=clean_env,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    logs.append(result.stdout)
    (BASE / "setup_execution.txt").write_text("\n".join(logs))
    if result.returncode:
        print(result.stdout[-6000:])
        raise RuntimeError(f"Command failed, exit {result.returncode}; see setup_execution.txt")
    return result.stdout

if not REPO.exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
    run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
assert run(["git", "rev-parse", "HEAD"], cwd=REPO).strip() == SOURCE_COMMIT
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True, system_site_packages=False).create(ENV)
run([PYTHON, "-I", "-m", "pip", "--isolated", "install", "--no-user", "-r",
     REPO / "configs/decision_topic_sample_requirements.txt"])
check = """
import importlib, importlib.metadata, json, pathlib, sys
expected = {"numpy": "2.5.4", "scipy": "1.18.1", "scikit-learn": "1.9.1"}
assert pathlib.Path(sys.prefix).resolve() == pathlib.Path(sys.argv[1]).resolve()
assert sys.prefix != sys.base_prefix
report = {"executable": sys.executable, "prefix": sys.prefix, "packages": {}}
for name, version in expected.items():
    module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
    assert importlib.metadata.version(name) == version == module.__version__
    assert pathlib.Path(module.__file__).resolve().is_relative_to(pathlib.Path(sys.prefix).resolve())
    report["packages"][name] = {"version": version, "origin": module.__file__}
print(json.dumps(report, indent=2))
"""
runtime_json = run([PYTHON, "-I", "-c", check, ENV])
(BASE / "runtime_preflight.json").write_text(runtime_json)
print(runtime_json)
print("Source:", SOURCE_COMMIT)


## 3. Δείγμα 500 κριτικών χωρίς μοντέλα
Το script διαβάζει μόνο το ήδη δεσμευμένο CSV. Δεν στέλνει κριτικές έξω.
Η επανάληψη δεν αντικαθιστά σιωπηλά ένα υπάρχον δείγμα.

In [ ]:
run([PYTHON, "-I", REPO / "scripts/benchmark_decision_topics.py", "sample", "--output", OUT], cwd=REPO)
manifest = json.loads((OUT / "sample_manifest.json").read_text())
assert manifest["sample_size"] == 500
assert manifest["split_counts"] == {"train": 300, "dev": 100, "test": 100}
print("Κριτικές:", manifest["sample_size"])
print("Train / dev / test:", manifest["split_counts"])
print("Β — ανεξάρτητες κοινές κρίσεις:", manifest["second_reviewer_size"])
print("Ξενοδοχεία:", manifest["hotel_counts"])


## 4. Κατέβασμα ZIP
Περιλαμβάνει τα κενά φύλλα, τις ελληνικές οδηγίες, το πρωτόκολλο και τα hashes.
Κράτησε το αρχικό ZIP αυτούσιο. Συμπλήρωσε αντίγραφα των CSV.

In [ ]:
import hashlib
import zipfile
from google.colab import files

package = BASE / "decision_topics_sample.zip"
with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.iterdir()):
        archive.write(path, "sample/" + path.name)
    for name in ("LABELING_EL.md", "PROTOCOL.md"):
        archive.write(REPO / "docs/experiments/decision_topic_benchmark" / name, name)
    archive.write(BASE / "runtime_preflight.json", "runtime_preflight.json")
    archive.write(BASE / "setup_execution.txt", "setup_execution.txt")
print("ZIP SHA-256:", hashlib.sha256(package.read_bytes()).hexdigest())
files.download(str(package))


## 5. Τι συμπληρώνεις

| Στήλη | Τι γράφεις |
| --- | --- |
| `label` | `cleanliness`, `service`, `location`, `value` ή `other` — ένα κύριο θέμα |
| `multiple_topics` | `yes` αν αξιολογούνται περισσότερα θέματα, αλλιώς `no` |
| `done` | `yes` αφού ολοκληρώσεις τη γραμμή |
| `note` | προαιρετική σημείωση για συζήτηση |

Μην αλλάξεις `review_id`, `review_sha256`, `review_text` ή τις επικεφαλίδες.
Η θετική κριτική έχει επίσης κύριο θέμα· δεν γράφεις `positive` / `negative`.
Μια βλάβη κλιματιστικού είναι `other`, η κακή ανταπόκριση προσωπικού είναι `service`.
Ακολούθησε τον κανόνα κύριου θέματος και ισοπαλίας στο **LABELING_EL.md** του ZIP.

Ο Α συμπληρώνει 500, ο Β τα 100 ανεξάρτητα πριν δει του Α.
Μετά συζητάτε τις διαφωνίες και συμπληρώνετε το `labels_final.csv` (500).
Σώσε ως `completed_A.csv`, `completed_B.csv`, `completed_final.csv` και στείλε
τα μαζί με το αρχικό ZIP. Τότε ελέγχουμε τα labels και τρέχουμε πρώτα dev,
μετά ένα παγωμένο test. Το 0,7 είναι δοκιμαστικό όριο παραπομπής σε άνθρωπο,
όχι εγγύηση σωστής απάντησης.